# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library and referencing all entities via their `@id` fields.

### Dataset Source
The dataset is sourced through a Croissant schema URL and supports machine-actionable, FAIR principles-aligned data loading.

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading

We load metadata and records from the dataset using `mlcroissant`. All objects in the Croissant specification (record sets, fields, columns) are referenced by their `@id`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Set the Croissant metadata schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"
# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(metadata["name"] + "\n" + metadata["description"])

## 2. Data Overview

List available record sets and their fields with their `@id`s. The `mlcroissant` API exposes them as Python objects accessible via `dataset.metadata.record_sets`.

Below we show all record sets, each field, and columns, referencing their `@id` for each.

In [ ]:
# Print all available record sets, fields, and columns by their @id

record_sets = dataset.metadata.record_sets
print(f"Record Sets Found: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record set @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'No name')} | Description: {rs.get('description', '')}")
    print(f"  Fields:")
    for field in rs.get('fields', []):
        print(f"    Field @id: {field['@id']} | Name: {field.get('name', '')}")
        for col in field.get('columns', []):
            print(f"      Column @id: {col['@id']} | Name: {col.get('name', '')}")
    print('----')

## 3. Data Extraction

We now extract data from a record set using its `@id` and load it as a Pandas DataFrame. Below, select the main record set(s) for analysis.


In [ ]:
# Identify main record set(s) to load. Usually this is the main survey data record set.
# We'll auto-select the first record set (customize if needed after reviewing output above).

record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded records for record_set @id={record_set_id}: {df.shape}")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")
        continue

# For demonstration, use the first successfully loaded main record set
main_record_set_id = record_set_ids[0] if len(record_set_ids) else None
if main_record_set_id and main_record_set_id in dataframes:
    main_df = dataframes[main_record_set_id]
    print(f"Columns in {main_record_set_id}:\n", main_df.columns.tolist())
    display(main_df.head())
else:
    main_df = None

## 4. Exploratory Data Analysis (EDA)

We'll demonstrate basic filtering, normalization, and grouping on one or more fields, referencing all by their `@id` (i.e., DataFrame column names as assigned by the Croissant schema and loaded by mlcroissant).

Suppose we use the PHQ-9 score field to identify high depressive symptoms and group by gender or education level (adjust as appropriate for your dataset schema).

In [ ]:
if main_df is not None:
    # Find numeric fields in the DataFrame; here we try the common PHQ-9 or GAD-7 fields
    numeric_field_ids = [col for col in main_df.columns if main_df[col].dtype in [np.float64, np.int64, float, int]]
    print("Numeric fields: ", numeric_field_ids)
    # Try to pick 'phq9_total_score' or similar, otherwise pick any
    preferred_fields = [c for c in main_df.columns if 'phq9' in c or 'gad7' in c]
    if preferred_fields:
        numeric_field_id = preferred_fields[0]
    elif numeric_field_ids:
        numeric_field_id = numeric_field_ids[0]
    else:
        print("No numeric fields detected for EDA.")
        numeric_field_id = None

    # Set threshold depending on what variable is picked; for PHQ-9 use 10
    threshold = 10
    if numeric_field_id:
        filtered_df = main_df[main_df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold} (N={len(filtered_df)}):")
        display(filtered_df.head())
        # Normalization
        norm_col = f"{numeric_field_id}_normalized"
        # Only compute if >1 row and std>0
        if len(filtered_df)>1 and filtered_df[numeric_field_id].std()>0:
            filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, norm_col]].head())
        # Group by a demographic field
        candidate_group_fields = [c for c in main_df.columns if ('gender' in c.lower()) or ('education' in c.lower()) or ('age' in c.lower())]
        group_field = candidate_group_fields[0] if candidate_group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field}:")
            display(grouped_df)
        else:
            print("No demographic field for grouping detected.")
    else:
        print("No numeric or psychometric score field found for EDA.")
else:
    print("No data loaded for EDA.")

## 5. Visualization

Visualize distribution of scores (e.g., PHQ-9, GAD-7) and their relationship with demographic fields. 
If columns were not detected properly above, adjust to use the correct `@id` from the earlier overview.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and numeric_field_id:
    plt.figure(figsize=(6,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        plt.figure(figsize=(7,4))
        sns.boxplot(data=main_df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} grouped by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No suitable numeric field found for visualization.")

## 6. Conclusion

We loaded the dataset via the Croissant schema and inspected its structure by referencing all record sets, fields, and columns by `@id`. We also demonstrated data extraction, basic exploratory processing (filtering, normalizing, grouping), and visualization for one of the survey's psychometric fields.

For reproducible and robust data analysis, always refer to the field/column or record set `@id` as the source of truth in FAIR and Croissant-aligned workflows. For more advanced use, explore `mlcroissant`'s support for relationships, value constraints, and metadata querying.